# 1. Install Dependencies
Installs Hugging Face datasets, PyTorch, and python-chess.


In [ ]:
!pip install datasets torch python-chess tqdm


# 2. Feature Extraction
Converts a FEN into our 768-element bitboard array.


In [ ]:
import chess
import torch
from datasets import load_dataset
import math
from tqdm import tqdm

def fen_to_features(fen):
    board = chess.Board(fen)
    features = torch.zeros(768, dtype=torch.float32)
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is not None:
            pt = piece.piece_type - 1 
            c = piece.color 
            idx = (int(c) * 6 + pt) * 64 + square
            features[idx] = 1.0
    return features


# 3. The NNUE Model
Defines the 768->256->32->32->1 architecture with ReLU activations and Sigmoid output.


In [ ]:
import torch.nn as nn

class NNUE(nn.Module):
    def __init__(self):
        super(NNUE, self).__init__()
        self.fc1 = nn.Linear(768, 256)
        self.fc2 = nn.Linear(256, 32)
        self.fc3 = nn.Linear(32, 32)
        self.fc4 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x


# 4. GPU Streaming & Training Loop
Streams the Lichess dataset in chunks directly to the GPU so Colab's RAM doesn't crash.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = NNUE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Load Dataset via Streaming
print("Connecting to Hugging Face...")
dataset = load_dataset("Lichess/chess-position-evaluations", split="train", streaming=True)

batch_size = 8192
num_batches_to_train = 2000  # Adjust this to train longer! (2000 = ~16 Million positions)

print("Starting training loop...")
model.train()

X_batch = []
y_batch = []

batch_idx = 0
progress_bar = tqdm(total=num_batches_to_train, desc="Training Batches")

for row in dataset:
    fen = row['fen']
    cp = row['cp']
    mate = row['mate']
    
    # Calculate Win/Draw/Loss probability directly
    if mate is not None:
        wdl = 1.0 if mate > 0 else 0.0
    elif cp is not None:
        wdl = 1.0 / (1.0 + math.pow(10.0, -cp / 400.0))
    else:
        continue  # Skip rows with no evaluation
        
    X_batch.append(fen_to_features(fen))
    y_batch.append([wdl])
    
    if len(X_batch) == batch_size:
        X_tensor = torch.stack(X_batch).to(device)
        y_tensor = torch.tensor(y_batch, dtype=torch.float32).to(device)
        
        optimizer.zero_grad()
        output = model(X_tensor)
        loss = criterion(output, y_tensor)
        loss.backward()
        optimizer.step()
        
        X_batch = []
        y_batch = []
        batch_idx += 1
        
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
        if batch_idx >= num_batches_to_train:
            break

progress_bar.close()
print("Training Complete!")


# 5. Export to .bin and Download
Runs the Post-Training Quantization (PTQ) scaling and downloads the `nnue_weights.bin` file to your PC.


In [ ]:
import struct
from google.colab import files

def export_nnue(model, filename="nnue_weights.bin"):
    QA = 255
    QB = 64
    
    # Layer 1: weights and biases are QA-scaled (int16)
    fc1_w = torch.round(model.fc1.weight.data * QA).clamp(-32768, 32767).to(torch.int16).flatten().tolist()
    fc1_b = torch.round(model.fc1.bias.data * QA).clamp(-32768, 32767).to(torch.int16).tolist()
    
    # Layer 2: weights are QB-scaled (int8), biases are QA*QB-scaled (int32)
    # Bias scale = QA*QB because: input is QA-scaled, weight is QB-scaled,
    # so the product sum is QA*QB-scaled. Bias must match before we divide by QA.
    fc2_w = torch.round(model.fc2.weight.data * QB).clamp(-128, 127).to(torch.int8).flatten().tolist()
    fc2_b = torch.round(model.fc2.bias.data * (QA * QB)).to(torch.int32).tolist()
    
    # Layer 3: weights are QB-scaled (int8), biases are QB*QB-scaled (int32)
    # Bias scale = QB*QB because: input is QB-scaled, weight is QB-scaled,
    # so the product sum is QB*QB-scaled. Bias must match before we divide by QB.
    fc3_w = torch.round(model.fc3.weight.data * QB).clamp(-128, 127).to(torch.int8).flatten().tolist()
    fc3_b = torch.round(model.fc3.bias.data * (QB * QB)).to(torch.int32).tolist()
    
    # Layer 4 (output): same logic as layer 3.
    # Input is QB-scaled, weight is QB-scaled, product is QB*QB-scaled.
    fc4_w = torch.round(model.fc4.weight.data * QB).clamp(-128, 127).to(torch.int8).flatten().tolist()
    fc4_b = torch.round(model.fc4.bias.data * (QB * QB)).to(torch.int32).tolist()

    with open(filename, 'wb') as f:
        f.write(struct.pack(f'<{len(fc1_w)}h', *fc1_w))
        f.write(struct.pack(f'<{len(fc1_b)}h', *fc1_b))
        
        f.write(struct.pack(f'<{len(fc2_w)}b', *fc2_w))
        f.write(struct.pack(f'<{len(fc2_b)}i', *fc2_b))
        
        f.write(struct.pack(f'<{len(fc3_w)}b', *fc3_w))
        f.write(struct.pack(f'<{len(fc3_b)}i', *fc3_b))
        
        f.write(struct.pack(f'<{len(fc4_w)}b', *fc4_w))
        f.write(struct.pack(f'<{len(fc4_b)}i', *fc4_b))
        
    print(f"Exported to {filename} ({len(fc1_w)*2 + len(fc1_b)*2 + len(fc2_w) + len(fc2_b)*4 + len(fc3_w) + len(fc3_b)*4 + len(fc4_w) + len(fc4_b)*4} bytes)")

model.cpu()
export_nnue(model, "nnue_weights.bin")
files.download("nnue_weights.bin")
